# AI401 — Expert Systems Lab
## Experiment 3: Agent Architectures — The BDI Deliberation Cycle

**Aim:**  
To implement a Belief-Desire-Intention (BDI) agent deliberation cycle in a dynamic Tileworld environment, analyze intention reconsideration frequency ($\gamma$), commitment strategies (single-minded vs open-minded), and evaluate resource-bounded deliberation under battery constraints.


---
### Core Implementation: Environment, BDI Architecture & Utilities
Implements Tileworld simulation environment, Belief Revision Function (BRF), Option Generator, Deliberation Filter, and Means-Ends Planner.


In [ ]:
import math
import random
import os
import sys
from PIL import Image, ImageDraw, ImageFont

class DynamicHole:
    def __init__(self, position: tuple, lifetime: int):
        self.position = position
        self.remaining_lifetime = lifetime

    def tick(self, count: int = 1) -> bool:
        self.remaining_lifetime -= count
        return self.remaining_lifetime <= 0


class Tileworld:
    def __init__(self, width: int = 15, height: int = 15, 
                 p_appear: float = 0.03, min_lifetime: int = 20, max_lifetime: int = 50,
                 world_speed: int = 1, seed: int = None):
        self.width = width
        self.height = height
        self.p_appear = p_appear
        self.min_lifetime = min_lifetime
        self.max_lifetime = max_lifetime
        self.world_speed = world_speed
        
        if seed is not None:
            random.seed(seed)
            
        self.all_grid_positions = set((x, y) for x in range(width) for y in range(height))
        self.holes = {}
        self.agent_pos = (7, 7)
        self.charger_pos = (0, 0)
        self.world_clock = 0
        self.total_holes_appeared = 0
        self.total_holes_filled = 0

    def step_world_clock(self, ticks: int = 1):
        for _ in range(ticks):
            self.world_clock += 1
            expired_keys = [pos for pos, hole in self.holes.items() if hole.tick(1)]
            for pos in expired_keys:
                del self.holes[pos]

            empty_positions = self.all_grid_positions - set(self.holes.keys()) - {self.agent_pos, self.charger_pos}
            num_empty = len(empty_positions)
            if num_empty > 0:
                num_spawn = sum(1 for _ in range(num_empty) if random.random() < self.p_appear)
                if num_spawn > 0:
                    spawned_pos = random.sample(list(empty_positions), min(num_spawn, num_empty))
                    for pos in spawned_pos:
                        lifetime = random.randint(self.min_lifetime, self.max_lifetime)
                        self.holes[pos] = DynamicHole(pos, lifetime)
                        self.total_holes_appeared += 1

    def get_percept(self, battery: int = 40) -> dict:
        return {
            "agent_pos": self.agent_pos,
            "holes": {pos: h.remaining_lifetime for pos, h in self.holes.items()},
            "charger_pos": self.charger_pos,
            "battery": battery,
            "world_clock": self.world_clock
        }

    def move_agent(self, new_pos: tuple) -> bool:
        self.agent_pos = new_pos
        hole_filled = False
        if new_pos in self.holes:
            del self.holes[new_pos]
            self.total_holes_filled += 1
            hole_filled = True
        return hole_filled

def manhattan_distance(pos1: tuple, pos2: tuple) -> int:
    return abs(pos1[0] - pos2[0]) + abs(pos1[1] - pos2[1])

def generate_plan(start: tuple, target: tuple) -> list:
    if start == target:
        return []
    path = []
    curr_x, curr_y = start
    target_x, target_y = target
    
    while curr_x != target_x:
        curr_x += 1 if target_x > curr_x else -1
        path.append((curr_x, curr_y))
    while curr_y != target_y:
        curr_y += 1 if target_y > curr_y else -1
        path.append((curr_x, curr_y))
    return path


---
### BDI Agent Class & Deliberation Cycle
Implements Belief Revision (`brf`), Desires (`options`), Intentions (`filter`), Means-Ends Planning, and Intention Reconsideration (Algorithm 2).


In [ ]:
class BDIAgent:
    def __init__(self, commitment_strategy: str = "single-minded", 
                 gamma: str = "1", resource_bounded: bool = False):
        self.commitment_strategy = commitment_strategy
        if isinstance(gamma, str) and gamma.isdigit():
            self.gamma = int(gamma)
        else:
            self.gamma = gamma
            
        self.resource_bounded = resource_bounded
        
        self.B = {
            "agent_pos": (7, 7),
            "holes": {},
            "charger_pos": (0, 0),
            "battery": 40,
            "world_clock": 0
        }
        self.battery_max = 40
        self.intention = None
        self.plan = []
        self.reconsideration_counter = 0
        self.deliberation_steps = 0
        self.stranded_steps = 0
        self.event_trace = []

    def log_event(self, step: int, event_type: str, details: str):
        self.event_trace.append({
            "step": step,
            "event": event_type,
            "details": details
        })

    def brf(self, percept: dict):
        self.B["agent_pos"] = percept["agent_pos"]
        self.B["holes"] = dict(percept["holes"])
        self.B["charger_pos"] = percept["charger_pos"]
        self.B["battery"] = percept["battery"]
        self.B["world_clock"] = percept["world_clock"]

    def options(self, B: dict, I: tuple) -> list:
        active_holes = list(B["holes"].keys())
        if not self.resource_bounded:
            return active_holes
        
        agent_pos = B["agent_pos"]
        charger_pos = B["charger_pos"]
        battery = B["battery"]
        
        valid_options = []
        for hole in active_holes:
            cost_to_hole = manhattan_distance(agent_pos, hole)
            cost_to_charger = manhattan_distance(hole, charger_pos)
            if cost_to_hole + cost_to_charger <= battery:
                valid_options.append(hole)
        
        if not valid_options or battery < 15:
            valid_options.append("RECHARGE")
            
        return valid_options

    def filter(self, B: dict, D: list, current_I: tuple) -> tuple:
        if not D:
            return None
        
        if "RECHARGE" in D and (B["battery"] < 15 or len(D) == 1):
            return "RECHARGE"
            
        agent_pos = B["agent_pos"]
        hole_options = [opt for opt in D if opt != "RECHARGE"]
        if not hole_options:
            return "RECHARGE"
            
        hole_options.sort(key=lambda h: manhattan_distance(agent_pos, h))
        best_option = hole_options[0]
        
        if self.commitment_strategy == "single-minded":
            if current_I in hole_options:
                return current_I
            return best_option
            
        elif self.commitment_strategy == "open-minded":
            return best_option
            
        return best_option

    def plan_means_ends(self, B: dict, I: tuple) -> list:
        if I is None:
            return []
        agent_pos = B["agent_pos"]
        target = B["charger_pos"] if I == "RECHARGE" else I
        return generate_plan(agent_pos, target)

def run_bdi_simulation(agent: BDIAgent, env: Tileworld, max_agent_steps: int = 600, trace_events: bool = False):
    agent_step = 0
    percept = env.get_percept(agent.B["battery"])
    agent.brf(percept)
    agent.intention = None
    agent.plan = []
    
    while agent_step < max_agent_steps:
        percept = env.get_percept(agent.B["battery"])
        agent.brf(percept)
        
        if agent.resource_bounded and agent.B["battery"] <= 0 and agent.B["agent_pos"] != env.charger_pos:
            agent.stranded_steps += 1
            agent_step += 1
            env.step_world_clock(env.world_speed)
            continue

        if agent.B["agent_pos"] == env.charger_pos and agent.B["battery"] < agent.battery_max:
            agent.B["battery"] = agent.battery_max
            if agent.intention == "RECHARGE":
                agent.intention = None
                agent.plan = []
        
        # Step 3: Deliberation when idle
        if agent.intention is None or not agent.plan:
            agent.deliberation_steps += 1
            agent_step += 1
            env.step_world_clock(env.world_speed)
            
            D = agent.options(agent.B, agent.intention)
            new_I = agent.filter(agent.B, D, agent.intention)
            
            if new_I is not None:
                agent.intention = new_I
                agent.plan = agent.plan_means_ends(agent.B, agent.intention)
                agent.reconsideration_counter = 0
                if trace_events:
                    agent.log_event(agent_step, "COMMIT", f"Committed to intention {agent.intention} (Plan length: {len(agent.plan)})")
            continue

        # Step 4: Execute move action
        if agent.plan:
            next_pos = agent.plan.pop(0)
            agent.reconsideration_counter += 1
            agent_step += 1
            
            if agent.resource_bounded:
                agent.B["battery"] -= 1
                
            hole_filled = env.move_agent(next_pos)
            env.step_world_clock(env.world_speed)
            
            percept = env.get_percept(agent.B["battery"])
            agent.brf(percept)
            
            # Step 5: Check Achievement
            achieved = False
            if agent.intention == "RECHARGE" and agent.B["agent_pos"] == env.charger_pos:
                achieved = True
            elif isinstance(agent.intention, tuple) and (hole_filled or agent.B["agent_pos"] == agent.intention):
                achieved = True
                
            if achieved:
                if trace_events:
                    agent.log_event(agent_step, "ACHIEVE", f"Achieved intention {agent.intention}")
                agent.intention = None
                agent.plan = []
                continue

            # Step 6: Algorithm 2 — Reconsideration
            should_reconsider = False
            if agent.gamma == "bold":
                should_reconsider = False
            elif isinstance(agent.gamma, int):
                if agent.reconsideration_counter >= agent.gamma:
                    should_reconsider = True

            if should_reconsider and agent_step < max_agent_steps:
                agent.reconsideration_counter = 0
                agent.deliberation_steps += 1
                agent_step += 1
                env.step_world_clock(env.world_speed)
                
                percept = env.get_percept(agent.B["battery"])
                agent.brf(percept)
                
                unachievable = False
                if isinstance(agent.intention, tuple) and agent.intention not in agent.B["holes"]:
                    unachievable = True
                    
                if unachievable:
                    if trace_events:
                        agent.log_event(agent_step, "DROP_UNACHIEVABLE", f"Dropped unachievable intention {agent.intention}")
                    agent.intention = None
                    agent.plan = []
                    continue

                D = agent.options(agent.B, agent.intention)
                preferred_option = agent.filter(agent.B, D, agent.intention)
                
                if preferred_option != agent.intention:
                    if trace_events:
                        agent.log_event(agent_step, "SWITCH_OPTION", f"Switched intention from {agent.intention} to preferred {preferred_option}")
                    agent.intention = preferred_option
                    agent.plan = agent.plan_means_ends(agent.B, agent.intention)

    effectiveness = env.total_holes_filled / env.total_holes_appeared if env.total_holes_appeared > 0 else 0.0
    return {
        "holes_filled": env.total_holes_filled,
        "holes_appeared": env.total_holes_appeared,
        "effectiveness": effectiveness,
        "deliberation_steps": agent.deliberation_steps,
        "stranded_steps": agent.stranded_steps,
        "event_trace": agent.event_trace
    }


---
### Sample Event Trace Demonstration
Generating an execution trace showing COMMIT, ACHIEVE, DROP_UNACHIEVABLE, and SWITCH_OPTION events.


In [ ]:
print("=== Generating Sample BDI Event Trace ===")
env = Tileworld(width=15, height=15, p_appear=0.04, world_speed=2, seed=42)
agent = BDIAgent(commitment_strategy="open-minded", gamma=2, resource_bounded=False)

results = run_bdi_simulation(agent, env, max_agent_steps=120, trace_events=True)

print(f"Total Holes Appeared: {results['holes_appeared']}")
print(f"Total Holes Filled:   {results['holes_filled']}")
print(f"Effectiveness:        {results['effectiveness']:.4f}\n")
print("Event Trace Snapshot (First 15 Events):")
print("-" * 75)
for event in results["event_trace"][:15]:
    print(f"Step {event['step']:3d} | Event: {event['event']:18s} | Details: {event['details']}")
print("-" * 75)


---
### Exercise 1: Parameter Sweep over Boldness ($\gamma$) vs World Dynamism
Evaluating agent effectiveness across $\gamma \in \{1, 2, 4, 8, 	ext{bold}\}$ and world speed $\in \{1, 2, 4, 8\}$.


In [ ]:
gammas = [1, 2, 4, 8, "bold"]
world_speeds = [1, 2, 4, 8]
num_seeds = 15
steps_per_run = 500

print("Sweeping Reconsideration Interval gamma vs World Dynamism...")
results_table = {}

for w in world_speeds:
    results_table[w] = {}
    for g in gammas:
        effs = []
        for seed in range(num_seeds):
            env = Tileworld(width=15, height=15, p_appear=0.03, world_speed=w, seed=1000 + seed)
            agent = BDIAgent(commitment_strategy="single-minded", gamma=g, resource_bounded=False)
            res = run_bdi_simulation(agent, env, max_agent_steps=steps_per_run)
            effs.append(res["effectiveness"])
        avg_eff = sum(effs) / num_seeds
        results_table[w][g] = {"effectiveness": avg_eff}

print("\nEffectiveness Matrix (Rows: World Speed, Cols: gamma):")
header = f"{'W_speed':<10} | " + " | ".join([f"{'gamma=' + str(g):<12}" for g in gammas])
print("-" * len(header))
print(header)
print("-" * len(header))
for w in world_speeds:
    row_str = f"{w:<10} | " + " | ".join([f"{results_table[w][g]['effectiveness']:<12.4f}" for g in gammas])
    print(row_str)
print("-" * len(header))


---
### Exercise 2: Resource-Bounded Deliberation vs Battery-Blind Agent
Comparing an agent with battery-aware filtering against a blind agent that runs out of battery.


In [ ]:
num_seeds = 20
steps_per_run = 500
world_speed = 2

smart_filled, smart_stranded, smart_appeared = [], [], []
blind_filled, blind_stranded, blind_appeared = [], [], []

for seed in range(num_seeds):
    # Smart Agent
    env_smart = Tileworld(width=15, height=15, p_appear=0.03, world_speed=world_speed, seed=2000 + seed)
    agent_smart = BDIAgent(commitment_strategy="single-minded", gamma=2, resource_bounded=True)
    res_smart = run_bdi_simulation(agent_smart, env_smart, max_agent_steps=steps_per_run)
    smart_filled.append(res_smart["holes_filled"])
    smart_appeared.append(res_smart["holes_appeared"])
    smart_stranded.append(res_smart["stranded_steps"])
    
    # Blind Agent
    env_blind = Tileworld(width=15, height=15, p_appear=0.03, world_speed=world_speed, seed=2000 + seed)
    agent_blind = BDIAgent(commitment_strategy="single-minded", gamma=2, resource_bounded=False)
    agent_blind.resource_bounded = True
    def blind_options(B, I):
        holes = list(B["holes"].keys())
        return holes if holes else ["RECHARGE"]
    agent_blind.options = blind_options
    res_blind = run_bdi_simulation(agent_blind, env_blind, max_agent_steps=steps_per_run)
    blind_filled.append(res_blind["holes_filled"])
    blind_appeared.append(res_blind["holes_appeared"])
    blind_stranded.append(res_blind["stranded_steps"])

avg_smart_filled = sum(smart_filled) / num_seeds
avg_smart_eff = avg_smart_filled / (sum(smart_appeared) / num_seeds)
avg_smart_stranded = sum(smart_stranded) / num_seeds

avg_blind_filled = sum(blind_filled) / num_seeds
avg_blind_eff = avg_blind_filled / (sum(blind_appeared) / num_seeds)
avg_blind_stranded = sum(blind_stranded) / num_seeds

print("=" * 80)
print("EXERCISE 2: RESOURCE-BOUNDED VS BATTERY-BLIND AGENT COMPARISON")
print("=" * 80)
print(f"1. Resource-Bounded BDI Agent | Holes Filled: {avg_smart_filled:5.1f} | Effectiveness: {avg_smart_eff:.4f} | Stranded Steps: {avg_smart_stranded:5.1f}")
print(f"2. Battery-Blind Agent        | Holes Filled: {avg_blind_filled:5.1f} | Effectiveness: {avg_blind_eff:.4f} | Stranded Steps: {avg_blind_stranded:5.1f}")
print("-" * 80)
print("Conclusion: Resource-bounded deliberation prevents agent starvation and sustains long-term effectiveness.")
